In [324]:
import pandas as pd
import numpy as np
import os
import re

from eval_plot_utils import parse_yml_pred,output_evaluation_outputs


In [325]:
pairs_aug = pd.read_csv('/ksi_ml/data/pairs_aug_ptmvar_flank.csv').iloc[:,1:].drop_duplicates()

tape_pred_wt = parse_yml_pred('/ksi_ml/code/evaluation/eval_outputs/exp_tape_bertbase_sep_v2_long__ptmvar_flank_wt__6eda71d75e35.pkl',
                          return_logits=True).drop_duplicates()

tape_pred_mut = parse_yml_pred('/ksi_ml/code/evaluation/eval_outputs/exp_tape_bertbase_sep_v2_long__ptmvar_flank_mut__6eda71d75e35.pkl',
                          return_logits=True).drop_duplicates()

In [326]:
kinome_dat = pd.read_pickle('/kinase_ml/data/human_kinome_seq_230301.pkl')

In [327]:
kinome_dat['p_aa'] = 'ST'
kinome_dat.loc[kinome_dat['pk_group']=='TK','p_aa'] = 'Y'

In [328]:
tape_pred['kinase_gene'].drop_duplicates().isin(kinome_dat['kinase_gene']).all()

np.True_

In [329]:
atypical_kinase = kinome_dat.loc[kinome_dat['pk_group']=='Atypical','kinase_gene'].tolist()

In [330]:
tape_pred_mut.columns = ['kinase_domain_seq','substrate_15aa_mut','probs_mut','logits_mut']
tape_pred_wt.columns = ['kinase_domain_seq','substrate_15aa_wt','probs_wt','logits_wt']

In [331]:
tape_pred = pd.merge(pairs_aug,tape_pred_wt,on = ['kinase_domain_seq','substrate_15aa_wt'])

In [332]:
tape_pred = pd.merge(tape_pred,tape_pred_mut,on = ['kinase_domain_seq','substrate_15aa_mut'])

In [333]:
tape_pred = tape_pred.drop_duplicates(['kinase','substrate_seq_mut','substrate_15aa_wt'])

In [334]:
tape_pred = tape_pred.loc[:, ['kinase','kinase_gene','substrate','p_ind','p_aa','mut_ind','mut_aa','substrate_15aa_wt','logits_wt','substrate_15aa_mut','logits_mut']]

In [806]:
15492*0.05

774.6

In [901]:
df = tape_pred

In [902]:
exclude_ls = ['CDC','CDK','ADCK','DYRK']
df = df.loc[df['kinase_gene'].apply(lambda s: np.all([elem not in s for elem in exclude_ls])),:]

df = df.loc[~df['kinase_gene'].isin(atypical_kinase),:]

In [904]:
import numpy as np
import pandas as pd

EVENT_KEY = ["p_ind", "p_aa", "mut_ind", "mut_aa", "substrate_15aa_wt", "substrate_15aa_mut"]

HIGH_RANK_CUTOFF = 300

def build_event_kinase_lists(
    df: pd.DataFrame,
    logit_col: str,                  # "logits_wt" or "logits_mut"
    kinase_col: str = "kinase_gene",
    low_logit_threshold: float = 5,
    top_frac: float = 0.01,
    suffix: str = "wt"               # "wt" or "mut" for column naming
) -> pd.DataFrame:
    """
    Build high/low kinase lists per EVENT_KEY, deduping at (EVENT_KEY + kinase).
    Ranking logic matches your original script (rank within kinase_gene across all rows).
    """
    cols = EVENT_KEY + [kinase_col, logit_col]
    d = df[cols].copy().dropna(subset=EVENT_KEY + [kinase_col, logit_col])

    # --- ranks within kinase_gene (global across dataset) ---
    d["logit_rank_within_kinase"] = (
        d.groupby(kinase_col)[logit_col]
         .rank(method="min", ascending=False)
    )
    d["n_rows_within_kinase"] = d.groupby(kinase_col)[logit_col].transform("size")
    # d["high_rank_cutoff"] = np.ceil(top_frac * d["n_rows_within_kinase"]).astype(int)
    d["high_rank_cutoff"] = HIGH_RANK_CUTOFF

    # --- low & high flags ---
    d["is_low_kinase"] = d[logit_col] < low_logit_threshold
    d["is_high_kinase"] = (
        (d["logit_rank_within_kinase"] <= d["high_rank_cutoff"]) &
        (~d["is_low_kinase"])
    )

    d["logit_category"] = np.select(
        [d["is_low_kinase"], d["is_high_kinase"]],
        ["low", "high"],
        default=""
    )

    # ------------------------------------------------------------
    # ✅ DEDUP at requested level: (EVENT_KEY + kinase)
    # Keep best observation for that kinase within that event
    # ------------------------------------------------------------
    d = d.sort_values(
        by=EVENT_KEY + [kinase_col, logit_col, "logit_rank_within_kinase"],
        ascending=[True]*len(EVENT_KEY) + [True, False, True]  # best = highest logit, then best rank
    ).drop_duplicates(subset=EVENT_KEY + [kinase_col], keep="first")

    # kinase:logit:rank string (now unique per event per kinase)
    d["kinase_logit_rank"] = (
        d[kinase_col].astype(str) + ":" +
        d[logit_col].round(3).astype(str) + ":" +
        d["logit_rank_within_kinase"].astype(int).astype(str)
    )

    def join_sorted(x: pd.Series) -> str:
        return "; ".join(
            x.sort_values(
                key=lambda s: s.str.extract(r":([-0-9]+)$")[0].astype(int),
                ascending=True
            )
        )

    # --- per-event stats (unique kinases already) ---
    stats = d.groupby(EVENT_KEY).agg(
        total_kinases=(kinase_col, "nunique"),
        mean_logits=(logit_col, "mean"),
        std_logits=(logit_col, "std"),
        min_logits=(logit_col, "min"),
        max_logits=(logit_col, "max"),
        high_rank_kinases=("is_high_kinase", lambda s: int(s.sum())),
        low_logit_kinases=("is_low_kinase", lambda s: int(s.sum())),
    ).reset_index()

    stats["high_kinase_ratio"] = stats["high_rank_kinases"] / stats["total_kinases"]
    stats["low_kinase_ratio"] = stats["low_logit_kinases"] / stats["total_kinases"]
    stats["logit_range"] = stats["max_logits"] - stats["min_logits"]

    # --- lists per event ---
    lists = (
        d.loc[d["logit_category"].isin(["high", "low"])]
         .groupby(EVENT_KEY + ["logit_category"])["kinase_logit_rank"]
         .apply(join_sorted)
         .unstack(fill_value="")
         .reset_index()
         .rename(columns={"high": f"high_kinases_{suffix}", "low": f"low_kinases_{suffix}"})
    )

    out = stats.merge(lists, on=EVENT_KEY, how="left")
    out[f"high_kinases_{suffix}"] = out[f"high_kinases_{suffix}"].fillna("")
    out[f"low_kinases_{suffix}"] = out[f"low_kinases_{suffix}"].fillna("")

    # rename stat columns to carry suffix
    out = out.rename(columns={
        "total_kinases": f"total_kinases_{suffix}",
        "mean_logits": f"mean_logits_{suffix}",
        "std_logits": f"std_logits_{suffix}",
        "min_logits": f"min_logits_{suffix}",
        "max_logits": f"max_logits_{suffix}",
        "high_rank_kinases": f"high_rank_kinases_{suffix}",
        "low_logit_kinases": f"low_logit_kinases_{suffix}",
        "high_kinase_ratio": f"high_kinase_ratio_{suffix}",
        "low_kinase_ratio": f"low_kinase_ratio_{suffix}",
        "logit_range": f"logit_range_{suffix}",
    })

    return out

# Build WT and MUT event-level summaries
wt_event = build_event_kinase_lists(df, logit_col="logits_wt", suffix="wt")
mut_event = build_event_kinase_lists(df, logit_col="logits_mut", suffix="mut")

# Merge back into df (one row per event_key will get same annotations)
df_out = (
    df.merge(wt_event, on=EVENT_KEY, how="left")
      .merge(mut_event, on=EVENT_KEY, how="left")
)


# ---------------------------------------------------------
# WT-high but MUT < 0, with KINASE:WT_LOGIT:WT_RANK
# Unique at EVENT_KEY + kinase
# ---------------------------------------------------------
tmp = df[EVENT_KEY + ["kinase_gene", "logits_wt", "logits_mut"]].dropna()

# WT ranking logic (same as before)
tmp["wt_rank"] = (
    tmp.groupby("kinase_gene")["logits_wt"]
       .rank(method="min", ascending=False)
)

tmp["mut_rank"] = (
    tmp.groupby("kinase_gene")["logits_mut"]
       .rank(method="min", ascending=False)
)
tmp["wt_n"] = tmp.groupby("kinase_gene")["logits_wt"].transform("size")

tmp["wt_cut"] = HIGH_RANK_CUTOFF
tmp["wt_low"] = tmp["logits_wt"] < 5
tmp["is_high_wt"] = (tmp["wt_rank"] <= tmp["wt_cut"]) & (~tmp["wt_low"])

# ✅ dedup at EVENT_KEY + kinase_gene (keep best WT row)
tmp = tmp.sort_values(
    by=EVENT_KEY + ["kinase_gene", "logits_wt", "wt_rank"],
    ascending=[True]*len(EVENT_KEY) + [True, False, True]
).drop_duplicates(subset=EVENT_KEY + ["kinase_gene"], keep="first")

# build string KINASE:WT_LOGIT:WT_RANK
tmp["kinase_logit_rank_wt"] = (
    tmp["kinase_gene"].astype(str) + ":" +
    tmp["logits_wt"].round(3).astype(str) + ":" +
    tmp["wt_rank"].astype(int).astype(str)
)

tmp["kinase_logit_rank_mut"] = (
    tmp["kinase_gene"].astype(str) + ":" +
    tmp["logits_mut"].round(3).astype(str) + ":" +
    tmp["mut_rank"].astype(int).astype(str)
)
# aggregate per event (sorted by WT rank ascending)
# flip = (
#     tmp.loc[tmp["is_high_wt"] & (tmp["logits_mut"] < 0)]
#       .groupby(EVENT_KEY)["kinase_logit_rank_mut"]
#       .apply(lambda s: "; ".join(
#           s.sort_values(
#               key=lambda ser: ser.str.extract(r":([-0-9]+)$")[0].astype(int),
#               ascending=True
#           )
#       ))
#       .reset_index(name="kinases_wtHigh_mutLt0")
# )


flip = (
    tmp.loc[tmp["is_high_wt"]]
      .groupby(EVENT_KEY)["kinase_logit_rank_mut"]
      .apply(lambda s: "; ".join(
          s.sort_values(
              key=lambda ser: ser.str.extract(r":([-0-9]+)$")[0].astype(int),
              ascending=True
          )
      ))
      .reset_index(name="kinases_wtHigh_mutLt0")
)

In [905]:
# merge back
df_out = df_out.merge(flip, on=EVENT_KEY, how="left")
df_out["kinases_wtHigh_mutLt0"] = df_out["kinases_wtHigh_mutLt0"].fillna("")

In [906]:
result_out = df_out.loc[(df_out['kinases_wtHigh_mutLt0']!='')&
    (df_out['high_kinase_ratio_wt']<0.05),:].loc[:,[
    "substrate", "p_ind", "p_aa","mut_aa",
    "mut_ind",
    "substrate_15aa_wt","substrate_15aa_mut",'high_kinase_ratio_wt','high_kinases_wt','kinases_wtHigh_mutLt0']].drop_duplicates(['substrate_15aa_wt','substrate_15aa_mut'])

In [907]:
substrate_stats = pd.merge(pairs_aug.loc[:,['substrate_15aa_wt','substrate_15aa_mut','dbSNP','FTID','substrate_gene']].drop_duplicates(),
                           result_out,
                           on = ['substrate_15aa_wt','substrate_15aa_mut'])

In [908]:
# Filter for substrates with few high-scoring and many low-scoring kinases
# CRITICAL: Must have at least 1 high-scoring kinase (> 0)
selective_substrates = substrate_stats.copy()

In [909]:
import requests
import pandas as pd
from typing import List, Dict
import time

def query_rsid_diseases(rsids: List[str], api: str = 'myvariant') -> pd.DataFrame:
    """
    Query disease associations for a list of rsIDs.
    
    Parameters:
    -----------
    rsids : List[str]
        List of rsIDs to query (e.g., ["rs958068", "rs121913400"])
    api : str
        Which API to use: 'myvariant' (default) or 'ensembl'
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: rsID, Clinical_Significance, Disease/Phenotype, 
        Review_Status, Accession
    """
    
    if api == 'myvariant':
        return _query_myvariant(rsids)
    elif api == 'ensembl':
        return _query_ensembl(rsids)
    else:
        raise ValueError("api must be 'myvariant' or 'ensembl'")


def _query_myvariant(rsids: List[str]) -> pd.DataFrame:
    """Query using MyVariant.info API"""
    base_url = "https://myvariant.info/v1/query"
    results = []
    
    for rsid in rsids:
        # Ensure rsid has 'rs' prefix
        if not rsid.startswith('rs'):
            rsid = f'rs{rsid}'
        
        try:
            # Query MyVariant.info
            response = requests.get(
                base_url,
                params={
                    'q': f'dbsnp.rsid:{rsid}',
                    'fields': 'dbsnp.rsid,clinvar,dbsnp.gene'
                },
                timeout=10
            )
            
            if response.status_code == 200:
                data = response.json()
                
                # Handle single hit or multiple hits
                hits = data.get('hits', [])
                if not hits:
                    results.append({
                        'rsID': rsid,
                        'Clinical_Significance': 'No data found',
                        'Disease/Phenotype': 'N/A',
                        'Review_Status': 'N/A',
                        'Accession': 'N/A'
                    })
                    continue
                
                # Process each hit (variant)
                found_clinvar = False
                for hit in hits:
                    clinvar = hit.get('clinvar', {})
                    
                    if not clinvar:
                        continue
                    
                    found_clinvar = True
                    
                    # Handle both single entry and list of entries
                    if isinstance(clinvar, dict):
                        clinvar_entries = [clinvar]
                    else:
                        clinvar_entries = clinvar if isinstance(clinvar, list) else []
                    
                    for entry in clinvar_entries:
                        # Extract RCV information
                        rcv = entry.get('rcv', {})
                        
                        # RCV can also be a list
                        if isinstance(rcv, list):
                            rcv_list = rcv
                        else:
                            rcv_list = [rcv] if rcv else []
                        
                        for rcv_entry in rcv_list:
                            if not isinstance(rcv_entry, dict):
                                continue
                            
                            # Extract clinical significance
                            significance = rcv_entry.get('clinical_significance', 'Unknown')
                            
                            # Extract conditions/diseases
                            conditions = rcv_entry.get('conditions', {})
                            if isinstance(conditions, dict):
                                phenotype = conditions.get('name', 'Not specified')
                            elif isinstance(conditions, list):
                                phenotype = '; '.join([c.get('name', '') for c in conditions if isinstance(c, dict) and c.get('name')])
                                if not phenotype:
                                    phenotype = 'Not specified'
                            else:
                                phenotype = 'Not specified'
                            
                            # Get review status
                            review_status = rcv_entry.get('review_status', 'Not available')
                            
                            # Get accession
                            accession = rcv_entry.get('accession', 'N/A')
                            
                            results.append({
                                'rsID': rsid,
                                'Clinical_Significance': significance,
                                'Disease/Phenotype': phenotype,
                                'Review_Status': review_status,
                                'Accession': accession
                            })
                
                if not found_clinvar:
                    # No ClinVar data but variant exists
                    gene_info = hits[0].get('dbsnp', {}).get('gene', {})
                    gene_name = gene_info.get('symbol', 'Unknown') if isinstance(gene_info, dict) else 'Unknown'
                    
                    results.append({
                        'rsID': rsid,
                        'Clinical_Significance': 'No ClinVar data',
                        'Disease/Phenotype': f'Gene: {gene_name}',
                        'Review_Status': 'N/A',
                        'Accession': 'N/A'
                    })
            else:
                results.append({
                    'rsID': rsid,
                    'Clinical_Significance': f'Query failed (status {response.status_code})',
                    'Disease/Phenotype': 'N/A',
                    'Review_Status': 'N/A',
                    'Accession': 'N/A'
                })
        
        except Exception as e:
            results.append({
                'rsID': rsid,
                'Clinical_Significance': f'Error: {str(e)[:100]}',
                'Disease/Phenotype': 'N/A',
                'Review_Status': 'N/A',
                'Accession': 'N/A'
            })
        
        # Be respectful to the API
        time.sleep(0.3)
    
    # Create DataFrame
    df = pd.DataFrame(results)
    return df


def _query_ensembl(rsids: List[str]) -> pd.DataFrame:
    """Query using Ensembl REST API (alternative method)"""
    base_url = "https://rest.ensembl.org/variation/human"
    results = []
    
    for rsid in rsids:
        # Ensure rsid has 'rs' prefix
        if not rsid.startswith('rs'):
            rsid = f'rs{rsid}'
        
        try:
            response = requests.get(
                f"{base_url}/{rsid}",
                headers={"Content-Type": "application/json"},
                params={"phenotypes": 1},
                timeout=10
            )
            
            if response.status_code == 200:
                data = response.json()
                
                # Check for phenotype associations
                phenotypes = data.get('phenotypes', [])
                
                if phenotypes:
                    for pheno in phenotypes:
                        # Extract relevant information
                        trait = pheno.get('trait', 'Unknown')
                        source = pheno.get('source', 'Unknown')
                        clinical_significance = pheno.get('clinical_significance', 'Unknown')
                        
                        results.append({
                            'rsID': rsid,
                            'Clinical_Significance': clinical_significance if clinical_significance else 'Not specified',
                            'Disease/Phenotype': trait,
                            'Review_Status': 'N/A',
                            'Accession': f"Source: {source}"
                        })
                else:
                    results.append({
                        'rsID': rsid,
                        'Clinical_Significance': 'No phenotype data',
                        'Disease/Phenotype': 'N/A',
                        'Review_Status': 'N/A',
                        'Accession': 'N/A'
                    })
            else:
                results.append({
                    'rsID': rsid,
                    'Clinical_Significance': f'Query failed (status {response.status_code})',
                    'Disease/Phenotype': 'N/A',
                    'Review_Status': 'N/A',
                    'Accession': 'N/A'
                })
        
        except Exception as e:
            results.append({
                'rsID': rsid,
                'Clinical_Significance': f'Error: {str(e)[:100]}',
                'Disease/Phenotype': 'N/A',
                'Review_Status': 'N/A',
                'Accession': 'N/A'
            })
        
        # Be respectful to the API
        time.sleep(0.3)
    
    df = pd.DataFrame(results)
    return df


def group_by_rsid(df: pd.DataFrame) -> pd.DataFrame:
    """
    Group DataFrame by rsID, concatenating all associated information.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame from query_rsid_diseases function
    
    Returns:
    --------
    pd.DataFrame
        Grouped DataFrame with one row per rsID
    """
    if df.empty:
        return df
    
    def concat_unique(series):
        """Concatenate unique non-N/A values"""
        unique_vals = series[~series.isin(['N/A', 'Not available', 'Unknown', 'Not specified'])].unique()
        if len(unique_vals) == 0:
            return 'N/A'
        return ' | '.join(unique_vals)
    
    grouped = df.groupby('rsID').agg({
        'Clinical_Significance': concat_unique,
        'Disease/Phenotype': concat_unique,
        'Review_Status': concat_unique,
        'Accession': concat_unique
    }).reset_index()
    
    return grouped
    

In [910]:
rsids = substrate_stats.loc[substrate_stats['dbSNP']!='-','dbSNP'].drop_duplicates().tolist()

In [911]:
print("Querying with MyVariant.info API...")
df = query_rsid_diseases(rsids, api='myvariant')

Querying with MyVariant.info API...


In [912]:
df_grouped = group_by_rsid(df)

In [913]:
selective_substrates = pd.merge(selective_substrates,df_grouped,left_on = 'dbSNP',right_on = 'rsID',how = 'left')

In [914]:
import requests
import pandas as pd
from typing import List, Tuple, Optional
import time

def query_omnipath_kinase_substrate(
    gene_pairs: List[Tuple[str, str]],
    effect_filter: Optional[str] = None
) -> pd.DataFrame:
    """
    Query OmniPath for kinase-substrate interactions matching the exact web interface format.
    
    Parameters:
    -----------
    gene_pairs : List[Tuple[str, str]]
        List of (source_gene, target_gene) tuples
        Example: [("GSK3B", "CTNNB1")]
    effect_filter : Optional[str]
        Filter by effect type: 'inhibition', 'stimulation', 'both', or None for all
    
    Returns:
    --------
    pd.DataFrame
        DataFrame matching OmniPath web interface format with columns:
        id, source, target, sourceGenesymbol, targetGenesymbol, isDirected, 
        isStimulation, isInhibition, consensusDirection, consensusStimulation,
        consensusInhibition, sources
    """
    
    base_url = "https://omnipathdb.org/interactions"
    
    all_results = []
    
    for source_gene, target_gene in gene_pairs:
        try:
            params = {
                'sources': source_gene.upper(),
                'targets': target_gene.upper(),
                'organisms': '9606',
                'genesymbols': '1',
                'datasets': 'omnipath',
                'license': 'academic'
            }
            
            response = requests.get(
                base_url,
                params=params,
                timeout=15
            )
            
            if response.status_code == 200:
                lines = response.text.strip().split('\n')
                
                if len(lines) <= 1:
                    continue
                
                # Parse header - first line contains column names
                header = lines[0].split('\t')
                
                # Parse each interaction
                for line in lines[1:]:
                    if not line.strip():
                        continue
                    
                    values = line.split('\t')
                    # Pad values if needed
                    values.extend([''] * (len(header) - len(values)))
                    
                    row_dict = dict(zip(header, values))
                    
                    # Build result with all available fields
                    result = {}
                    
                    # Map common field names
                    field_mapping = {
                        'source_genesymbol': 'sourceGenesymbol',
                        'target_genesymbol': 'targetGenesymbol',
                        'is_directed': 'isDirected',
                        'is_stimulation': 'isStimulation',
                        'is_inhibition': 'isInhibition',
                        'consensus_direction': 'consensusDirection',
                        'consensus_stimulation': 'consensusStimulation',
                        'consensus_inhibition': 'consensusInhibition',
                        'type':'type'
                    }
                    
                    # Copy all fields, applying mapping where available
                    for key, value in row_dict.items():
                        display_key = field_mapping.get(key, key)
                        result[display_key] = value
                    
                    # Convert 1/0 to TRUE/FALSE for boolean-looking columns
                    bool_columns = ['isDirected', 'isStimulation', 'isInhibition', 
                                  'consensusDirection', 'consensusStimulation', 'consensusInhibition',
                                  'is_directed', 'is_stimulation', 'is_inhibition',
                                  'consensus_direction', 'consensus_stimulation', 'consensus_inhibition']
                    
                    for col in bool_columns:
                        if col in result:
                            result[col] = 'TRUE' if result[col] == '1' else 'FALSE'
                    
                    # Apply effect filter if specified
                    if effect_filter:
                        consensus_inhib = result.get('consensusInhibition') or result.get('consensus_inhibition', 'FALSE')
                        consensus_stimul = result.get('consensusStimulation') or result.get('consensus_stimulation', 'FALSE')
                        
                        if effect_filter.lower() == 'inhibition' and consensus_inhib != 'TRUE':
                            continue
                        elif effect_filter.lower() == 'stimulation' and consensus_stimul != 'TRUE':
                            continue
                        elif effect_filter.lower() == 'both' and (consensus_inhib != 'TRUE' or consensus_stimul != 'TRUE'):
                            continue
                    
                    all_results.append(result)
            
            else:
                print(f"Query failed for {source_gene}->{target_gene}: status {response.status_code}")
        
        except Exception as e:
            print(f"Error querying {source_gene}->{target_gene}: {str(e)}")
        
        time.sleep(0.5)
    
    if not all_results:
        return pd.DataFrame()
    
    df = pd.DataFrame(all_results)
    
    # Prioritize camelCase column names for display
    # If we have both formats, use camelCase
    rename_map = {}
    for old_name in ['source_genesymbol', 'target_genesymbol', 'is_directed', 
                     'is_stimulation', 'is_inhibition', 'consensus_direction',
                     'consensus_stimulation', 'consensus_inhibition','type']:
        camel_name = old_name.replace('_', ' ').title().replace(' ', '')
        camel_name = camel_name[0].lower() + camel_name[1:]  # Make first letter lowercase
        if old_name in df.columns and camel_name not in df.columns:
            rename_map[old_name] = camel_name
    
    if rename_map:
        df = df.rename(columns=rename_map)
    
    return df

In [915]:
pairs_aug.columns

Index(['kinase_gene', 'kinase_name', 'pk_group', 'pk_family', 'pk_subfamily',
       'kinase', 'kinase_seq', 'kinseq_len', 'kinase_coord',
       'kinase_domain_seq', 'substrate_gene', 'substrate', 'wt_aa', 'mut_ind',
       'mut_aa', 'var_type', 'var_class', 'var_dist', 'dbSNP', 'FTID', 'p_ind',
       'substrate_seq', 'label', 'substrate_seq_mut', 'substrate_15aa_wt',
       'substrate_15aa_mut', 'p_aa'],
      dtype='object')

In [916]:
kinase_dict = (
    tape_pred.loc[:, ['kinase_gene', 'kinase']]
    .drop_duplicates()
    .groupby('kinase_gene')['kinase']
    .apply(lambda s: list(s)[0])
    .to_dict())

substrate_dict =  (
    pairs_aug.loc[:, ['substrate_gene', 'substrate']]
    .drop_duplicates()
    .groupby('substrate_gene')['substrate']
    .apply(lambda s: list(s)[0])
    .to_dict())

In [917]:
len(substrate_stats['substrate_gene'].drop_duplicates().tolist())

1347

In [918]:
# Query GSK3B -> CTNNB1 interactions
from tqdm.auto import tqdm
all_df = []
for substrate in tqdm(substrate_stats['substrate_gene'].drop_duplicates().tolist()):
    substrate_stats_test = substrate_stats.loc[substrate_stats['substrate_gene']==substrate,:]
    source = substrate_dict[substrate]
    df_curr_allkinase = query_omnipath_kinase_substrate([(source,source)])
    for i,row in substrate_stats_test.iterrows(): 
        all_kinases = row['high_kinases_wt']
        if all_kinases!='':
            kinase_ls = all_kinases.split('; ')
            for kinase_logit in kinase_ls:
                target = kinase_dict[kinase_logit.split(':')[0]]
                if df_curr_allkinase.shape[0]>0:
                    df_curr_allkinase_1 = df_curr_allkinase.loc[(df_curr_allkinase['source']==source)&(df_curr_allkinase['target']==target),:]
                    df_curr_allkinase_2 = df_curr_allkinase.loc[(df_curr_allkinase['source']==target)&(df_curr_allkinase['target']==source),:]
                    all_df.append(df_curr_allkinase_1)
                    all_df.append(df_curr_allkinase_2)
                    

  0%|          | 0/1347 [00:00<?, ?it/s]

Error querying Q9H2D6->Q9H2D6: HTTPSConnectionPool(host='omnipathdb.org', port=443): Max retries exceeded with url: /interactions?sources=Q9H2D6&targets=Q9H2D6&organisms=9606&genesymbols=1&datasets=omnipath&license=academic (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x79885bf94d10>, 'Connection to omnipathdb.org timed out. (connect timeout=15)'))
Error querying Q9UJA5->Q9UJA5: HTTPSConnectionPool(host='omnipathdb.org', port=443): Max retries exceeded with url: /interactions?sources=Q9UJA5&targets=Q9UJA5&organisms=9606&genesymbols=1&datasets=omnipath&license=academic (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x79885ae1cbd0>, 'Connection to omnipathdb.org timed out. (connect timeout=15)'))
Error querying Q14679->Q14679: HTTPSConnectionPool(host='omnipathdb.org', port=443): Max retries exceeded with url: /interactions?sources=Q14679&targets=Q14679&organisms=9606&genesymbols=1&datasets=omnipath&license=academic (Caused by

In [919]:
all_df = pd.concat(all_df)

In [920]:
omnipath_df = all_df.drop_duplicates(['source','target'])

In [921]:
omnipath_df.to_pickle('./260121_omnipath_query_flank_hits.pkl')

In [922]:
# start clean
ss = selective_substrates.copy()

# ensure we don't have duplicate index issues
ss = ss.reset_index(drop=True)

# split to list
ss["high_kinase_list"] = ss["high_kinases_wt"].fillna("").str.split(";")

# explode into long form
ss_long = ss.explode("high_kinase_list", ignore_index=True)

# extract kinase symbol (before first ':')
ss_long["high_kinase"] = (
    ss_long["high_kinase_list"]
    .astype(str)
    .str.strip()
    .replace("", pd.NA)
    .str.split(":")
    .str[0]
)

ss_long = ss_long.dropna(subset=["high_kinases_wt"])

In [923]:
ss_long = (
    selective_substrates
    .reset_index(drop=True)
    .assign(high_kinase=lambda d: d["high_kinases_wt"].fillna("").str.split(";"))
    .explode("high_kinase", ignore_index=True)
)

ss_long["high_kinase"] = (
    ss_long["high_kinase"]
    .astype(str)
    .str.strip()
    .replace("", pd.NA)
    .str.split(":")
    .str[0]
)

ss_long = ss_long.dropna(subset=["high_kinase"])

In [924]:
op = omnipath_df.copy()

# forward edges
op_fwd = op.assign(
    kinase=op["sourceGenesymbol"],
    kinase_acc=op["source"],
    substrate=op["targetGenesymbol"],
    substrate_acc=op["target"],
    direction="source_to_target"
)

# reverse edges
op_rev = op.assign(
    kinase=op["targetGenesymbol"],
    kinase_acc=op["target"],
    substrate=op["sourceGenesymbol"],
    substrate_acc=op["source"],
    direction="target_to_source"
)

op_edges = pd.concat([op_fwd, op_rev], ignore_index=True)

In [925]:
ss_long['high_kinase_acc'] = ss_long['high_kinase'].apply(lambda s: kinase_dict[s])
ss_long['substrate_acc'] = ss_long['substrate_gene'].apply(lambda s: substrate_dict[s])

In [926]:
kinome_dat.columns = ['kinase_gene', 'kinase_name', 'pk_group', 'pk_family', 'pk_subfamily',
       'kinase', 'kinase_seq', 'kinseq_len', 'kinase_coord', 'p_aa_thresh']

In [927]:
ss_long = pd.merge(ss_long,kinome_dat.loc[:,['kinase','p_aa_thresh']],left_on = 'high_kinase_acc',right_on = 'kinase')

In [928]:
ss_long = ss_long.loc[((ss_long['p_aa']=='Y')&(ss_long['p_aa_thresh']=='Y')|(ss_long['p_aa']!='Y')&(ss_long['p_aa_thresh']!='Y')),:]

kinase column predicted to phosphorylate the phosphosite in each row defined by p_ind and substrate column, the interactions are filtered to include confidently interacting proteins documented in omnipath, and mutating into a non-phosphorylable amino acid results in the dbSNP column  rsID, which suggests disease consequences in 'Clinical_Significance', 'Disease/Phenotype' columns. for each substrate_seq_15aa, summarize the impact, and try to support the kinase-substrate-disease linkage with literature searches, ignore other columns 

In [929]:
ss_melt = pd.merge(tape_pred.loc[:,['substrate_15aa_wt','kinase_gene','substrate_15aa_mut','logits_wt','logits_mut']].drop_duplicates(),
         ss_long,left_on = ['substrate_15aa_wt','kinase_gene','substrate_15aa_mut'],
         right_on = ['substrate_15aa_wt','high_kinase','substrate_15aa_mut'])

In [930]:
ss_melt['logit_delta'] = ss_melt['logits_wt'] - ss_melt['logits_mut']

In [931]:
ss_melt = ss_melt.loc[ss_melt['logit_delta']>2.5,:]

In [932]:
ss_melt = ss_melt.loc[ss_melt['dbSNP']!='-',:]
ss_melt = ss_melt.loc[ss_melt['Clinical_Significance'].apply(lambda s: 'pathogenic' in s.lower()),:]

In [933]:
ss_melt_overlap_omnipath = ss_melt.merge(
    op_edges,
    left_on=["high_kinase_acc", "substrate_acc"],
    right_on=["kinase_acc", "substrate_acc"],
    how="inner"
)

In [934]:
n = ss_melt_overlap_omnipath.drop_duplicates(['substrate_15aa_wt','kinase_gene']).shape[0]
print(f'number of substrate-kinase changes in flanking state with omnipath record of interaction: {n}')

number of substrate-kinase changes in flanking state with omnipath record of interaction: 30


In [953]:
ss_melt_novel = ss_melt.loc[(ss_melt['logit_delta']>2.5),:]

In [959]:
pairs_aug_phosphonetwork = pd.read_csv('../../data/pairs_aug_phosphonetwork.csv').iloc[:,1:]

In [954]:
ss_melt_novel = ss_melt_novel.loc[~ss_melt_novel['substrate_gene'].apply(lambda s:'KRT' in s),:]
ss_melt_novel = ss_melt_novel.loc[~ss_melt_novel['kinase_gene'].apply(lambda s:'DYRK' in s),:]

In [955]:
# ss_melt_novel_stats = ss_melt_novel.drop_duplicates(['substrate_gene','kinase_gene','p_ind']).groupby('substrate_gene').count().reset_index().sort_values('substrate_15aa_wt')

In [956]:
# ss_melt_novel = ss_melt_novel.loc[~ss_melt_novel['substrate_gene'].isin(ss_melt_novel_stats.loc[ss_melt_novel_stats['substrate_15aa_wt']>5,'substrate_gene']),:]

In [949]:
ss_melt_novel_ks = ss_melt_novel.drop_duplicates(['substrate_gene','kinase_gene']).loc[:,['kinase_gene','substrate_gene','kinase','substrate','Clinical_Significance',
                                                                    'Disease/Phenotype']].drop_duplicates()

In [1018]:
ss_melt_novel_phosphonetwork = pd.merge(ss_melt_novel,pairs_aug_phosphonetwork.loc[:,['kinase','substrate']].drop_duplicates())
# ss_melt_novel_phosphonetwork = ss_melt_novel_phosphonetwork.loc[~ss_melt_novel_phosphonetwork['Clinical_Significance'].apply(lambda s: 'conflicting' in s.lower()),:]

In [978]:
import requests
import time
import sys
from typing import List, Tuple, Dict, Optional

class StringDBQuery:
    """Class to handle STRING-DB API queries for kinase-substrate pairs"""
    
    def __init__(self, species: int = 9606, version: str = "12.0"):
        """
        Initialize STRING-DB query handler
        
        Args:
            species: NCBI taxon ID (default: 9606 for human)
            version: STRING-DB version to use (default: "12.0")
        """
        self.species = species
        self.base_url = f"https://version-{version.replace('.', '-')}.string-db.org/api"
        self.caller_identity = "kinase_substrate_query"
        
    def map_uniprot_to_string(self, uniprot_ids: List[str]) -> Dict[str, str]:
        """
        Map UniProt accession IDs to STRING identifiers
        
        Args:
            uniprot_ids: List of UniProt accession IDs
            
        Returns:
            Dictionary mapping UniProt IDs to STRING IDs
        """
        print(f"Mapping {len(uniprot_ids)} UniProt IDs to STRING IDs...")
        
        request_url = f"{self.base_url}/tsv/get_string_ids"
        
        params = {
            "identifiers": "\r".join(uniprot_ids),
            "species": self.species,
            "limit": 1,  # Get only the best match
            "echo_query": 1,
            "caller_identity": self.caller_identity
        }
        
        response = requests.post(request_url, data=params)
        
        if response.status_code != 200:
            print(f"Error mapping IDs: HTTP {response.status_code}")
            return {}
        
        mapping = {}
        for line in response.text.strip().split("\n")[1:]:  # Skip header
            parts = line.split("\t")
            if len(parts) >= 3:
                query_id = parts[0]
                string_id = parts[2]
                mapping[query_id] = string_id
        
        print(f"Successfully mapped {len(mapping)}/{len(uniprot_ids)} IDs")
        return mapping
    
    def get_interaction(self, protein_a: str, protein_b: str) -> Optional[Dict]:
        """
        Get interaction information between two proteins
        
        Args:
            protein_a: STRING ID for first protein
            protein_b: STRING ID for second protein
            
        Returns:
            Dictionary with interaction data or None if no interaction exists
        """
        request_url = f"{self.base_url}/json/network"
        
        params = {
            "identifiers": f"{protein_a}\r{protein_b}",
            "species": self.species,
            "required_score": 0,  # Get all interactions regardless of score
            "caller_identity": self.caller_identity
        }
        
        response = requests.post(request_url, data=params)
        
        if response.status_code != 200:
            return None
        
        data = response.json()
        
        # Find the interaction between our two proteins
        for interaction in data:
            if (interaction['stringId_A'] == protein_a and interaction['stringId_B'] == protein_b) or \
               (interaction['stringId_A'] == protein_b and interaction['stringId_B'] == protein_a):
                return interaction
        
        return None
    
    def query_kinase_substrate_pairs(
        self, 
        pairs: List[Tuple[str, str]], 
        delay: float = 1.0
    ) -> List[Dict]:
        """
        Query STRING-DB for kinase-substrate pairs
        
        Args:
            pairs: List of (kinase_uniprot_id, substrate_uniprot_id) tuples
            delay: Delay between API calls in seconds (to avoid overloading server)
            
        Returns:
            List of dictionaries containing interaction results
        """
        # Collect all unique UniProt IDs
        all_uniprot_ids = list(set([p[0] for p in pairs] + [p[1] for p in pairs]))
        
        # Map UniProt IDs to STRING IDs
        uniprot_to_string = self.map_uniprot_to_string(all_uniprot_ids)
        
        results = []
        
        print(f"\nQuerying {len(pairs)} kinase-substrate pairs...")
        
        for i, (kinase_uniprot, substrate_uniprot) in enumerate(pairs, 1):
            # Get STRING IDs
            kinase_string = uniprot_to_string.get(kinase_uniprot)
            substrate_string = uniprot_to_string.get(substrate_uniprot)
            
            if not kinase_string or not substrate_string:
                result = {
                    'kinase_uniprot': kinase_uniprot,
                    'substrate_uniprot': substrate_uniprot,
                    'kinase_string': kinase_string,
                    'substrate_string': substrate_string,
                    'interaction_found': False,
                    'error': 'Failed to map one or both UniProt IDs to STRING'
                }
                results.append(result)
                continue
            
            # Query interaction
            interaction = self.get_interaction(kinase_string, substrate_string)
            
            if interaction:
                result = {
                    'kinase_uniprot': kinase_uniprot,
                    'substrate_uniprot': substrate_uniprot,
                    'kinase_string': kinase_string,
                    'kinase_name': interaction['preferredName_A'] if interaction['stringId_A'] == kinase_string else interaction['preferredName_B'],
                    'substrate_string': substrate_string,
                    'substrate_name': interaction['preferredName_B'] if interaction['stringId_A'] == kinase_string else interaction['preferredName_A'],
                    'interaction_found': True,
                    'combined_score': interaction['score'],
                    'neighborhood_score': interaction['nscore'],
                    'fusion_score': interaction['fscore'],
                    'cooccurrence_score': interaction['pscore'],
                    'coexpression_score': interaction['ascore'],
                    'experimental_score': interaction['escore'],
                    'database_score': interaction['dscore'],
                    'textmining_score': interaction['tscore']
                }
            else:
                result = {
                    'kinase_uniprot': kinase_uniprot,
                    'substrate_uniprot': substrate_uniprot,
                    'kinase_string': kinase_string,
                    'substrate_string': substrate_string,
                    'interaction_found': False,
                    'error': None
                }
            
            results.append(result)
            
            # Progress update
            if i % 10 == 0:
                print(f"Processed {i}/{len(pairs)} pairs...")
            
            # Delay to be polite to the server
            if i < len(pairs):
                time.sleep(delay)
        
        print(f"Completed querying {len(pairs)} pairs")
        return results


def print_results(results: List[Dict]):
    """Print results in a readable format"""
    
    print("\n" + "="*80)
    print("RESULTS SUMMARY")
    print("="*80)
    
    total = len(results)
    found = sum(1 for r in results if r['interaction_found'])
    
    print(f"\nTotal pairs queried: {total}")
    print(f"Interactions found: {found} ({100*found/total:.1f}%)")
    print(f"No interaction: {total - found} ({100*(total-found)/total:.1f}%)")
    
    print("\n" + "="*80)
    print("DETAILED RESULTS")
    print("="*80)
    
    for i, result in enumerate(results, 1):
        print(f"\n{i}. {result['kinase_uniprot']} → {result['substrate_uniprot']}")
        
        if result['interaction_found']:
            print(f"   ✓ Interaction found!")
            print(f"   Kinase: {result.get('kinase_name', 'N/A')} ({result['kinase_string']})")
            print(f"   Substrate: {result.get('substrate_name', 'N/A')} ({result['substrate_string']})")
            print(f"   Combined Score: {result['combined_score']}")
            print(f"   Experimental Score: {result['experimental_score']}")
            print(f"   Database Score: {result['database_score']}")
            print(f"   Textmining Score: {result['textmining_score']}")
        else:
            error_msg = result.get('error', 'No interaction found in STRING-DB')
            print(f"   ✗ {error_msg}")


def save_results_to_tsv(results: List[Dict], filename: str = "stringdb_results.tsv"):
    """Save results to a TSV file"""
    
    with open(filename, 'w') as f:
        # Write header
        headers = [
            'kinase_uniprot', 'substrate_uniprot',
            'kinase_string', 'substrate_string',
            'kinase_name', 'substrate_name',
            'interaction_found', 'combined_score',
            'experimental_score', 'database_score',
            'textmining_score', 'coexpression_score',
            'neighborhood_score', 'fusion_score',
            'cooccurrence_score', 'error'
        ]
        f.write('\t'.join(headers) + '\n')
        
        # Write data
        for result in results:
            row = [
                result.get('kinase_uniprot', ''),
                result.get('substrate_uniprot', ''),
                result.get('kinase_string', ''),
                result.get('substrate_string', ''),
                result.get('kinase_name', ''),
                result.get('substrate_name', ''),
                str(result.get('interaction_found', False)),
                str(result.get('combined_score', '')),
                str(result.get('experimental_score', '')),
                str(result.get('database_score', '')),
                str(result.get('textmining_score', '')),
                str(result.get('coexpression_score', '')),
                str(result.get('neighborhood_score', '')),
                str(result.get('fusion_score', '')),
                str(result.get('cooccurrence_score', '')),
                result.get('error', '')
            ]
            f.write('\t'.join(row) + '\n')
    
    print(f"\nResults saved to {filename}")



    

In [986]:
example_pairs = list(zip(ss_melt_novel_ks['kinase'],ss_melt_novel_ks['substrate']))
    
print("STRING-DB Kinase-Substrate Query Tool")
print("="*80)
print(f"Species: Human (NCBI Taxon ID: 9606)")
print(f"Example pairs to query: {len(example_pairs)}")

# Initialize query handler
query = StringDBQuery(species=9606)

# Query the pairs
results = query.query_kinase_substrate_pairs(example_pairs, delay=1.0)

STRING-DB Kinase-Substrate Query Tool
Species: Human (NCBI Taxon ID: 9606)
Example pairs to query: 614
Mapping 400 UniProt IDs to STRING IDs...
Successfully mapped 399/400 IDs

Querying 614 kinase-substrate pairs...
Processed 10/614 pairs...
Processed 20/614 pairs...
Processed 30/614 pairs...
Processed 40/614 pairs...
Processed 50/614 pairs...
Processed 60/614 pairs...
Processed 70/614 pairs...
Processed 80/614 pairs...
Processed 90/614 pairs...
Processed 100/614 pairs...
Processed 110/614 pairs...
Processed 120/614 pairs...
Processed 130/614 pairs...
Processed 140/614 pairs...
Processed 150/614 pairs...
Processed 160/614 pairs...
Processed 170/614 pairs...
Processed 180/614 pairs...
Processed 190/614 pairs...
Processed 200/614 pairs...
Processed 210/614 pairs...
Processed 220/614 pairs...
Processed 230/614 pairs...
Processed 240/614 pairs...
Processed 250/614 pairs...
Processed 260/614 pairs...
Processed 270/614 pairs...
Processed 280/614 pairs...
Processed 290/614 pairs...
Processed 

In [989]:
stringdb_df = pd.DataFrame(results)

In [996]:
stringdb_df_08 = stringdb_df.loc[(stringdb_df['combined_score']>0.8),:]

In [998]:
ss_melt_novel_stringdb = pd.merge(ss_melt_novel,stringdb_df_08,left_on = ['kinase','substrate'],
        right_on = ['kinase_uniprot','substrate_uniprot'])

In [1002]:
ss_melt_novel_stringdb.loc[:,['kinase_gene','substrate_gene','p_ind','p_aa','dbSNP','Clinical_Significance',
                                                                    'Disease/Phenotype']].drop_duplicates(['kinase_gene','substrate_gene','p_ind'])

,kinase_gene,substrate_gene,p_ind,p_aa,dbSNP,Clinical_Significance,Disease/Phenotype
0,AKT1,NCF1,45,T,rs119103270,Pathogenic,"Granulomatous disease, chronic, autosomal rece..."
1,AKT1,NF1,2188,S,rs267606598,Pathogenic | Uncertain significance,"Neurofibromatosis, type 1 (NF1) | Hereditary c..."
2,AurB,TP53,269,S,rs121913343,Pathogenic/Likely pathogenic | Uncertain signi...,Li-Fraumeni syndrome 1 (LFS) | Hereditary canc...
19,BUBR1,BRCA1,1212,S,rs1060502347,Uncertain significance | Conflicting interpret...,Hereditary breast ovarian cancer syndrome | He...
21,CaMK4,DCX,92,S,rs104894785,Pathogenic,Ectopic tissue | Subcortical laminar heterotop...
22,Erk1,CTNNB1,33,S,rs121913396,Pathogenic | Likely pathogenic,Pilomatrixoma (PTR) | not provided
30,Erk1,CTNNB1,37,S,rs121913396,Pathogenic | Likely pathogenic,Pilomatrixoma (PTR) | not provided
39,Erk1,RAF1,640,T,rs587777587,Uncertain significance | Pathogenic | Likely p...,LEOPARD syndrome 2 (LPRD2); Noonan syndrome 5 ...
40,Erk1,RPS6KA3,386,S,rs122454129,Pathogenic,"Intellectual disability, X-linked 19 (XLID19)"
41,Erk2,RAF1,640,T,rs587777587,Uncertain significance | Pathogenic | Likely p...,LEOPARD syndrome 2 (LPRD2); Noonan syndrome 5 ...


In [1010]:
ss_melt_overlap_omnipath_nr = ss_melt_overlap_omnipath.drop_duplicates(['substrate_15aa_wt','kinase_gene']).loc[:,['kinase_gene','substrate_gene','p_ind','p_aa','mut_ind','mut_aa','dbSNP','Clinical_Significance',
                                                                                                                   'Disease/Phenotype','substrate_15aa_wt','logits_wt','logits_mut']]

In [1012]:
ss_melt_overlap_omnipath_nr['db_source'] = 'omnipath'

In [1020]:
ss_melt_overlap_phosphonetwork_nr = ss_melt_novel_phosphonetwork.drop_duplicates(['substrate_15aa_wt','kinase_gene']).loc[:,['kinase_gene','substrate_gene','p_ind','p_aa','mut_ind','mut_aa','dbSNP','Clinical_Significance',
                                                                                                                   'Disease/Phenotype','substrate_15aa_wt','logits_wt','logits_mut']]

In [1022]:
ss_melt_overlap_phosphonetwork_nr['db_source'] = 'phosphonetwork'

In [1024]:
ss_melt_overlap_stringdb_nr = ss_melt_novel_stringdb.drop_duplicates(['substrate_15aa_wt','kinase_gene']).loc[:,['kinase_gene','substrate_gene','p_ind','p_aa','mut_ind','mut_aa','dbSNP','Clinical_Significance',
                                                                                                                   'Disease/Phenotype','substrate_15aa_wt','logits_wt','logits_mut']]

In [1026]:
ss_melt_overlap_stringdb_nr['db_source'] = 'stringdb'

In [1029]:
ss_melt_overlap_3dbs = pd.concat([ss_melt_overlap_omnipath_nr,ss_melt_overlap_phosphonetwork_nr,ss_melt_overlap_stringdb_nr])

In [1032]:
ss_melt_overlap_3dbs = ss_melt_overlap_3dbs.groupby(['kinase_gene', 'substrate_gene', 'p_ind', 'p_aa', 'mut_ind', 'mut_aa',
       'dbSNP', 'Clinical_Significance', 'Disease/Phenotype',
       'substrate_15aa_wt', 'logits_wt', 'logits_mut']).head(1)

In [1034]:
ss_melt_overlap_3dbs.to_csv('260128_ptmvar_flanks_overlap_dbs.csv')